In [1]:
#!pip install transformers datasets torch evaluate seqeval


In [2]:
import wandb
wandb.init(mode="disabled", project="qa_squad_demo")

In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments, pipeline
from datasets import load_dataset

# 1. Load the CoNLL-2003 dataset
dataset = load_dataset("conll2003", trust_remote_code=True)


In [4]:

# 2. Load the tokenizer and model
model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(dataset["train"].features["ner_tags"].feature.names))


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:

# 3. Prepare the data
label_list = dataset["train"].features["ner_tags"].feature.names


In [6]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,  # Set a fixed maximum length
        return_tensors="pt"
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

# 4. Set the format for PyTorch
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

In [7]:

# 5. Define evaluation metrics
import evaluate
metric = evaluate.load("seqeval")

def compute_metrics(predictions):
    preds, labels = predictions
    preds = preds.argmax(axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(preds, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(preds, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [17]:

# 6. Define training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",           # 模型輸出資料夾
    num_train_epochs=3,               # 訓練 3 個 epoch
    per_device_train_batch_size=16,   # 每個設備 batch size
    per_device_eval_batch_size=16,    # 評估時的 batch size
    learning_rate=2e-5,               # 學習率
    weight_decay=0.01,                # 權重衰減
    logging_dir="./logs",             # 日誌資料夾
    logging_steps=10,                 # 每 10 step log 一次
)


In [21]:

# 7. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


C:\Users\wayne\AppData\Local\Temp\ipykernel_13800\509930194.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [23]:

# 8. Fine-tune the model
trainer.train()


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss
10,1.703300
20,0.718000
30,0.591200
40,0.435000
50,0.358500
60,0.330500
70,0.206900
80,0.170500
90,0.180600
100,0.133800


TrainOutput(global_step=2634, training_loss=0.05213635281469438, metrics={'train_runtime': 569.8203, 'train_samples_per_second': 73.923, 'train_steps_per_second': 4.623, 'total_flos': 2751824963545344.0, 'train_loss': 0.05213635281469438, 'epoch': 3.0})

In [25]:

# 9. Evaluate the model
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")


Evaluation results: {'eval_loss': 0.03822962939739227, 'eval_precision': 0.9451413279812678, 'eval_recall': 0.951987870619946, 'eval_f1': 0.9485522450692405, 'eval_accuracy': 0.9913099390136976, 'eval_runtime': 13.1588, 'eval_samples_per_second': 246.982, 'eval_steps_per_second': 15.503, 'epoch': 3.0}


In [26]:

# 10. Save the fine-tuned model
model.save_pretrained("./fine_tuned_ner_model")
tokenizer.save_pretrained("./fine_tuned_ner_model")


('./fine_tuned_ner_model\\tokenizer_config.json',
 './fine_tuned_ner_model\\special_tokens_map.json',
 './fine_tuned_ner_model\\vocab.txt',
 './fine_tuned_ner_model\\added_tokens.json',
 './fine_tuned_ner_model\\tokenizer.json')

In [27]:

# 11. Inference using pipeline
ner_pipeline = pipeline("token-classification", model="./fine_tuned_ner_model", tokenizer="./fine_tuned_ner_model", aggregation_strategy="simple")

# Example text for inference
text = "Hugging Face Inc. is a company based in New York City."

# Get predictions
predictions = ner_pipeline(text)
print(f"Predictions: {predictions}")

Device set to use cuda:0


Predictions: [{'entity_group': 'LABEL_3', 'score': 0.9775865, 'word': 'Hu', 'start': 0, 'end': 2}, {'entity_group': 'LABEL_4', 'score': 0.9216988, 'word': '##gging Face Inc.', 'start': 2, 'end': 17}, {'entity_group': 'LABEL_0', 'score': 0.99987334, 'word': 'is a company based in', 'start': 18, 'end': 39}, {'entity_group': 'LABEL_5', 'score': 0.9988362, 'word': 'New', 'start': 40, 'end': 43}, {'entity_group': 'LABEL_6', 'score': 0.9960711, 'word': 'York City', 'start': 44, 'end': 53}, {'entity_group': 'LABEL_0', 'score': 0.99985933, 'word': '.', 'start': 53, 'end': 54}]


In [31]:
# 對照表：將模型輸出的 LABEL_x 轉換為實際標籤
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

# 替換 entity_group 為可讀標籤
for entity in predictions:
    label_idx = int(entity['entity_group'].split('_')[1])
    entity['entity_group'] = label_list[label_idx]

# 輸出結果
print("\n📌 Named Entities:")
for entity in predictions:
    if entity['entity_group'] != 'O':  # 只顯示實體
        print(f"- {entity['entity_group']}: {entity['word']} (score={entity['score']:.3f})")



📌 Named Entities:
- B-ORG: Hu (score=0.978)
- I-ORG: ##gging Face Inc. (score=0.922)
- B-LOC: New (score=0.999)
- I-LOC: York City (score=0.996)
